In [ ]:
import pandas as pd
import re
import unicodedata
from google.colab import files
# 1. Upload Files
print(" Train file upload(Final_Cleaned_Labels.xlsx):")
uploaded = files.upload()

print("\nTest file upload(test.xlsx):")
uploaded = files.upload()
def normalize_unicode(text):
    return unicodedata.normalize('NFC', str(text))

def replace_common_english(text):
    replacements = {
        "login": "লগইন",
        "refund": "রিফান্ড",
        "delivery": "ডেলিভারি",
        "account": "একাউন্ট",
        "service": "সার্ভিস",
        "customer": "কাস্টমার",
        "code": "কোড",
        "otp": "ওটিপি"
    }
    for k, v in replacements.items():
        text = re.sub(rf"\b{k}\b", v, text, flags=re.IGNORECASE)
    return text

def remove_noise(text):
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"\+?\d[\d -]{8,}\d", " ", text)
    return text

def keep_bangla_only(text):
    return re.sub(r"[^\u0980-\u09FF০-৯\s.,!?।]", " ", text)

def normalize_repetition(text):
    return re.sub(r'(.)\1{2,}', r'\1\1', text)

def normalize_punctuation(text):
    text = re.sub(r'!+', '!', text)
    text = re.sub(r'\?+', '?', text)
    text = re.sub(r'\.+', '.', text)
    return text

def normalize_spaces(text):
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_text(text):
    if pd.isna(text) or str(text).strip() == "":
        return ""
    text = normalize_unicode(text)
    text = replace_common_english(text)
    text = remove_noise(text)
    text = keep_bangla_only(text)
    text = normalize_repetition(text)
    text = normalize_punctuation(text)
    text = normalize_spaces(text)
    return text

# 3. Label Cleaning

VALID_LABELS = ['Billing', 'Refund', 'Delivery', 'Customer_Service',
                'Product_Quality', 'Account', 'Technical', 'Network']

def fix_label(label):
    label = label.strip()
    label_lower = label.lower().replace(' ', '_').replace('-', '_')
    for valid in VALID_LABELS:
        if label_lower == valid.lower():
            return valid
    return None

def fix_row_labels(label_str):
    if pd.isna(label_str) or str(label_str).strip() in ['', 'nan']:
        return None
    parts = re.split(r'[;,]', str(label_str))
    fixed = [fix_label(p) for p in parts if fix_label(p)]
    return ';'.join(fixed) if fixed else None

# 4. Load Data

train = pd.read_excel('Final_Cleaned_Labels__2_.xlsx')
test  = pd.read_excel('test__1_.xlsx')

# 5. Apply Preprocessing

train['text'] = train['text'].fillna("").apply(preprocess_text)
test['text']  = test['text'].fillna("").apply(preprocess_text)

# 6. Clean Labels

train['labels'] = train['labels'].apply(fix_row_labels)
test['labels']  = test['labels'].apply(fix_row_labels)

# 7. Remove empty & duplicates

train = train[train['text'].str.strip() != ""]
train = train.dropna(subset=['labels'])
train = train.drop_duplicates(subset='text')

test = test[test['text'].str.strip() != ""]
test = test.dropna(subset=['labels'])
test = test.drop_duplicates(subset='text')

train = train[['text', 'labels']]
test  = test[['text', 'labels']]

train.to_excel('train_preprocessed.xlsx', index=False)
test.to_excel('test_preprocessed.xlsx',  index=False)

print("Done!")
print(f"Train: {len(train)} samples")
print(f"Test:  {len(test)} samples")

from collections import Counter
print("\nTrain Label Distribution:")
all_labels = []
for l in train['labels']:
    all_labels.extend(l.split(';'))
for k, v in sorted(Counter(all_labels).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")

print("\nTest Label Distribution:")
all_labels = []
for l in test['labels']:
    all_labels.extend(l.split(';'))
for k, v in sorted(Counter(all_labels).items(), key=lambda x: -x[1]):
    print(f"  {k}: {v}")